# CP-final — Text-to-Speech: ≤15s Spoken Summaries with On-Screen Citations

**Owner:** Shane · **Deliverable:** Final — text-to-speech synthesis

This notebook turns the Answerer's structured payload (`speech`, `citations`, `comparison_table` — see `prompts/answerer_critic.md`) into a **spoken audio file**, using `src/rag/tts.py`:

* `estimate_speech_seconds(text)` / `fits_budget(text)` — enforce the ≤15s (~55-word) spoken-answer budget before synthesis.
* `speak(text, out_path=None, provider=None, voice=None)` — fragment-based synthesis to a finished `.wav`/`.mp3` file, dispatching to `pyttsx3` (offline default), `openai`, or `elevenlabs`.

It runs with **zero setup and zero API keys** — the default provider is `pyttsx3`, a fully offline system-TTS engine — and degrades gracefully to that default if a paid provider is requested without its key/package, the same pattern Clark's `web_search.py` uses for search providers.

## 0. Setup

In [1]:
import os, sys
sys.path.append(os.path.abspath('../src'))
from IPython.display import Audio

from rag.config import get_config
from rag import tts

cfg = get_config()
print('TTS provider :', cfg.tts_provider)
print('TTS voice    :', cfg.tts_voice or '(engine default)')
print('OPENAI_API_KEY set?     :', bool(os.environ.get('OPENAI_API_KEY')))
print('ELEVENLABS_API_KEY set? :', bool(os.environ.get('ELEVENLABS_API_KEY')))

TTS provider : pyttsx3
TTS voice    : (engine default)
OPENAI_API_KEY set?     : False
ELEVENLABS_API_KEY set? : False


## 1. Why fragment-based synthesis

There are two ways to turn text into spoken audio:

* **Streaming synthesis** — audio is generated and played back incrementally as tokens/words arrive, so playback can start before the full text exists.
* **Fragment-based synthesis** — the complete text is generated first, then synthesized to a finished audio file in one call, which is then played back or handed off as a whole.

**This notebook uses fragment-based synthesis.** The Answerer's `speech` field is not safe to speak until the Critic has verified it (grounding + safety, `prompts/answerer_critic.md`) — there is no partial prefix of the text that is safe to say early. Since the whole payload must exist and pass the Critic gate before anything is spoken, streaming buys nothing here, while fragment-based synthesis is simpler and works uniformly across every provider, including the fully offline `pyttsx3` default.

## 2. Load a sample Answerer payload

In production, this payload comes straight from the Answerer/Critic node (Victoria's part) after the Critic returns `action: "accept"` — see `prompts/answerer_critic.md`. Here we load it from the same fixture the prompt disclosure ships: `prompts/fewshots/answerer_examples.json`.

In [2]:
import json

with open('../prompts/fewshots/answerer_examples.json') as f:
    examples = json.load(f)

example = examples[0]
print('speech:\n ', example['output']['speech'])
print()
print('citations:')
for c in example['output']['citations']:
    print(' -', c['title'], '|', c['doc_id'], '|', c['url'])
print()
print('comparison_table:')
for row in example['output']['comparison_table']:
    print(' -', row)

speech:
  My top pick is the GreenGleam Steel-Safe Eco cleaner — plant-based, 4.6 stars, about $12.49 for 16 ounces. I compared it with a smaller NatureNest travel size. Details and sources are on your screen. Want the most affordable or the highest rated?

citations:
 - Steel-Safe Eco Stainless Steel Cleaner & Polish, 16 oz | 74d9f6149b125affab1f3b8d14798b0b | https://www.amazon.com/dp/B0SAMPLE000
 - NatureNest Stainless Steel Cleaner, 8 oz travel | b5bc95c5d69c5de186be4a4ef1a98ed1 | https://www.amazon.com/dp/B0SAMPLE018

comparison_table:
 - {'title': 'Steel-Safe Eco Stainless Steel Cleaner & Polish', 'price': 12.49, 'rating': 4.6, 'price_per_oz': 0.78, 'ingredients': 'plant-based surfactants, citric acid', 'doc_id': '74d9f6149b125affab1f3b8d14798b0b'}
 - {'title': 'NatureNest Stainless Steel Cleaner (travel)', 'price': 6.49, 'rating': 4.4, 'price_per_oz': 0.81, 'ingredients': 'coco-glucoside, citric acid', 'doc_id': 'b5bc95c5d69c5de186be4a4ef1a98ed1'}


## 3. Enforce the ≤15s / ≤~55-word budget

`prompts/answerer_critic.md` requires the spoken text to be ≤15 seconds (~2–3 sentences, ≤~55 words) and the Critic is supposed to reject anything longer. `tts.speak()` re-checks this defensively anyway — belt-and-suspenders — so a bug upstream never crashes synthesis, it just truncates and warns.

In [3]:
speech_text = example['output']['speech']
too_long_text = (
    "My top pick is the GreenGleam Steel-Safe Eco cleaner, a plant-based "
    "stainless steel cleaner and polish that costs about $12.49 for a 16 "
    "ounce bottle and has an average rating of 4.6 out of 5 stars from "
    "verified purchasers, and I also compared it against several other "
    "similar products including a smaller 8 ounce travel-sized option from "
    "a competing brand called NatureNest which costs quite a bit less per "
    "bottle but noticeably more per ounce once you account for the size, "
    "so let me know if you would like the full comparison table."
)

for label, text in [('fixture speech', speech_text), ('deliberately too long', too_long_text)]:
    secs = tts.estimate_speech_seconds(text)
    ok = tts.fits_budget(text)
    print(f'{label:22s} ~{secs:5.1f}s  {"PASS" if ok else "FAIL (will be truncated by speak())"}')

fixture speech         ~ 16.8s  FAIL (will be truncated by speak())
deliberately too long  ~ 37.2s  FAIL (will be truncated by speak())


## 4. Synthesize speech (fragment-based)

`speak()` writes a complete audio file and returns its path — the fragment-based approach from §1. With no `provider` argument and no `TTS_PROVIDER` env override, this uses the offline `pyttsx3` default from `cfg.tts_provider`.

In [4]:
out_path = tts.speak(speech_text)

duration_s = tts.estimate_speech_seconds(speech_text)
size_kb = out_path.stat().st_size / 1024
print('wrote     :', out_path)
print(f'~duration : {duration_s:.1f}s (estimated)')
print(f'file size : {size_kb:.1f} KB')

tts.speak: text is ~16.8s, over the 15s budget; truncating to the last full sentence that fits (~13.6s).


wrote     : /Users/ceverson/Development/Academic/ADSP_32028/final/rag-system/audio/summary_a345deddc1bd.wav
~duration : 16.8s (estimated)
file size : 633.1 KB


In [5]:
Audio(filename=str(out_path))

## 5. Align spoken audio with on-screen citations

`tts.speak()` only ever voices the Answerer's `speech` field — it never reads `citations[*].url`/`doc_id` or the full `comparison_table` aloud. This mirrors the system prompt's voice-style rule (`prompts/system_assistant.md`, "Voice style"): *"No markdown, emojis, URLs, or reading out long ingredient lists aloud — those belong on the screen, not in the speech."* Citations and the comparison table are rendered on screen, next to (not inside) the audio.

In [6]:
import pandas as pd

citations_df = pd.DataFrame(example['output']['citations'])
comparison_df = pd.DataFrame(example['output']['comparison_table'])

display(Audio(filename=str(out_path)))
print('Citations (on screen, not spoken):')
display(citations_df)
print('Comparison table (on screen, not spoken):')
display(comparison_df)

Citations (on screen, not spoken):


,doc_id,title,url,source
0,74d9f6149b125affab1f3b8d14798b0b,Steel-Safe Eco Stainless Steel Cleaner & Polis...,https://www.amazon.com/dp/B0SAMPLE000,private
1,b5bc95c5d69c5de186be4a4ef1a98ed1,"NatureNest Stainless Steel Cleaner, 8 oz travel",https://www.amazon.com/dp/B0SAMPLE018,private


Comparison table (on screen, not spoken):


,title,price,rating,price_per_oz,ingredients,doc_id
0,Steel-Safe Eco Stainless Steel Cleaner & Polish,12.49,4.6,0.78,"plant-based surfactants, citric acid",74d9f6149b125affab1f3b8d14798b0b
1,NatureNest Stainless Steel Cleaner (travel),6.49,4.4,0.81,"coco-glucoside, citric acid",b5bc95c5d69c5de186be4a4ef1a98ed1


## 6. Graceful degradation without an API key

If `provider="openai"` or `"elevenlabs"` is requested but the matching API key (or, for ElevenLabs, the package) isn't available, `speak()` logs a warning to stderr and falls back to `pyttsx3` instead of raising — the same graceful-degradation pattern as Clark's `web_search.py`.

In [7]:
os.environ.pop('OPENAI_API_KEY', None)  # ensure the key really is absent for this demo

fallback_path = tts.speak(speech_text, provider='openai')
print('wrote (via fallback):', fallback_path)
Audio(filename=str(fallback_path))

tts.speak: text is ~16.8s, over the 15s budget; truncating to the last full sentence that fits (~13.6s).
tts.speak: provider='openai' requested but OPENAI_API_KEY is not set; falling back to offline pyttsx3.


wrote (via fallback): /Users/ceverson/Development/Academic/ADSP_32028/final/rag-system/audio/summary_a345deddc1bd.wav


## 7. Batch demo across multiple queries

Every example in `answerer_examples.json` — including the ungrounded / empty-result case — should still produce a playable, on-budget clip.

In [8]:
manifest = []
for ex in examples:
    text = ex['output']['speech']
    path = tts.speak(text)
    manifest.append({
        'user_priority': ex['user_priority'],
        'n_citations': len(ex['output']['citations']),
        'est_seconds': round(tts.estimate_speech_seconds(text), 1),
        'fits_budget': tts.fits_budget(text),
        'audio_path': str(path),
        'size_kb': round(path.stat().st_size / 1024, 1),
    })

pd.DataFrame(manifest)

tts.speak: text is ~16.8s, over the 15s budget; truncating to the last full sentence that fits (~13.6s).


,user_priority,n_citations,est_seconds,fits_budget,audio_path,size_kb
0,budget + eco,2,16.8,False,/Users/ceverson/Development/Academic/ADSP_3202...,633.1
1,empty result,0,10.4,True,/Users/ceverson/Development/Academic/ADSP_3202...,307.2


---
### Handoff notes

* **Signature:** `speak(text: str, out_path: str | Path | None = None, provider: str | None = None, voice: str | None = None) -> Path`.
* **Output location:** defaults to `rag-system/audio/summary_<content-hash>.wav` (deterministic, no timestamp — repeat calls with the same text overwrite in place). `audio/query*.wav` (the ASR notebook's samples) are untouched and stay git-tracked; `audio/summary_*` is gitignored.
* **Optional orchestration:** `04_orchestration.ipynb` can call `rag.tts.speak(answer['speech'])` on the Answerer's accepted payload to attach spoken audio to a full agent turn — it is optional because the text/citations UI already satisfies the pipeline without audio.
* **Env vars:** `TTS_PROVIDER` (`pyttsx3` default | `openai` | `elevenlabs`), `TTS_VOICE`, `TTS_MODEL` (OpenAI TTS model id), plus `OPENAI_API_KEY` / `ELEVENLABS_API_KEY` only if you opt into a paid provider. Nothing is required to run this notebook as-is.